In [62]:
"""Bước 1: tự tính lan truyền tiến bằng numpy.

Chú thích viết tiếng Việt có dấu, phần in ra màn hình viết không dấu.
"""

import numpy as np


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def relu(z):
    return np.maximum(0, z)


# Phần 1: một nơ-ron với hàm sigmoid, chính là hồi quy logistic
w = np.array([0.8, -0.5, 0.3])
b = -0.2
x = np.array([2.0, 1.0, 3.0])
z = w @ x + b
print("Mot no-ron:")
print(f"  z = {w[0]}*{x[0]} + ({w[1]})*{x[1]} + {w[2]}*{x[2]}"
      f" + ({b}) = {z:.2f}")
print(f"  a = sigmoid(z) = {sigmoid(z):.4f}")
print()

# Phần 2: mạng 2 đầu vào, 2 nơ-ron ẩn ReLU, 1 nơ-ron ra
# Trọng số tự đặt để mạng tính đúng phép XOR
W1 = np.array([[1.0, 1.0],   # cột 1: trọng số vào nơ-ron ẩn h1
               [1.0, 1.0]])  # cột 2: trọng số vào nơ-ron ẩn h2
b1 = np.array([0.0, -1.0])
W2 = np.array([1.0, -2.0])
b2 = 0.0

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
print("Mang giai XOR:")
print("  x1 x2  h1 h2  y")
for xi in X:
    h = relu(xi @ W1 + b1)   # lớp ẩn
    y = h @ W2 + b2          # lớp ra
    print(f"  {xi[0]:.0f}  {xi[1]:.0f}   {h[0]:.0f}  {h[1]:.0f}  {y:.0f}")
print()

# Cả bốn đầu vào cùng lúc bằng phép nhân ma trận
H = relu(X @ W1 + b1)
Y = H @ W2 + b2
print("Tinh ca bang mot lan:", Y)

Mot no-ron:
  z = 0.8*2.0 + (-0.5)*1.0 + 0.3*3.0 + (-0.2) = 1.80
  a = sigmoid(z) = 0.8581

Mang giai XOR:
  x1 x2  h1 h2  y
  0  0   0  0  0
  0  1   1  0  1
  1  0   1  0  1
  1  1   2  1  0

Tinh ca bang mot lan: [0. 1. 1. 0.]


In [64]:
"""Bước 2: một nơ-ron không giải được XOR, một lớp ẩn thì được."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

df = pd.read_csv("xor.csv")   # file nằm ở thư mục gốc của Colab
print(df.head())              # xem tên cột; xóa dòng này khi đã ổn
print()

X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = LogisticRegression().fit(X_tr, y_tr)
print(f"Hoi quy logistic: {accuracy_score(y_te, lr.predict(X_te)):.4f}")
print()

# Dữ liệu nhỏ nên dùng solver="lbfgs", học nhanh và ổn định hơn
print("So no-ron an | so tham so | tap hoc | tap kiem tra")
for so in [1, 2, 4, 8]:
    mlp = MLPClassifier(hidden_layer_sizes=(so,), solver="lbfgs",
                        max_iter=5000, random_state=0)
    mlp.fit(X_tr, y_tr)
    # coefs_ là danh sách ma trận trọng số, intercepts_ là hệ số chặn
    so_ts = sum(w.size for w in mlp.coefs_)
    so_ts += sum(b.size for b in mlp.intercepts_)
    a_hoc = accuracy_score(y_tr, mlp.predict(X_tr))
    a_kt = accuracy_score(y_te, mlp.predict(X_te))
    print(f"  {so:2d}           | {so_ts:3d}        | {a_hoc:.4f}  | {a_kt:.4f}")

       x1      x2  nhan
0  0.2739 -0.4604     0
1 -0.9181 -0.9669     1
2  0.6265  0.8255     1
3  0.2133  0.4590     1
4  0.0872  0.8701     1

Hoi quy logistic: 0.4500

So no-ron an | so tham so | tap hoc | tap kiem tra
   1           |   5        | 0.6533  | 0.6900
   2           |   9        | 0.6533  | 0.6900
   4           |  17        | 0.9033  | 0.9300
   8           |  33        | 0.9167  | 0.9100


In [69]:

# BƯỚC 3: MẠNG TO THÌ HỌC VẸT, ĐIỀU CHỈNH ALPHA THÌ ĐỠ

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 1. Đọc dữ liệu
df = pd.read_csv("/content/hai_vang_trang.csv")

# 2. Chọn đặc trưng và nhãn
X = df[["x1", "x2"]]
y = df["nhan"]

# 3. Chia dữ liệu thành tập học và tập kiểm tra
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.25,
    random_state=17,
    stratify=y
)

# 4. Xây dựng mô hình MLP
def mang(lop_an, alpha):
    return make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=lop_an,
            alpha=alpha,
            solver="lbfgs",
            max_iter=20000,
            random_state=0
        )
    )

# 5. So sánh ba mạng nơ-ron
print("Ba mang             Tap hoc    Tap kiem tra")

for ten, lop_an, alpha in [
    ("(2,), alpha 1e-4", (2,), 1e-4),
    ("(100, 100), 1e-5", (100, 100), 1e-5),
    ("(100, 100), 1.0", (100, 100), 1.0)
]:
    m = mang(lop_an, alpha).fit(X_tr, y_tr)

    print(
        f"{ten:20s}"
        f"{accuracy_score(y_tr, m.predict(X_tr)):.4f}    "
        f"{accuracy_score(y_te, m.predict(X_te)):.4f}"
    )

print()

# 6. Chọn alpha bằng kiểm định chéo trên tập học
print("Alpha kiem dinh cheo")

diem = {}

for alpha in [1e-5, 1e-3, 0.1, 1.0, 3.0, 10.0]:
    diem[alpha] = cross_val_score(
        mang((100, 100), alpha),
        X_tr,
        y_tr,
        cv=5
    ).mean()

    print(f"{alpha:<8} {diem[alpha]:.4f}")

# 7. Chọn alpha tốt nhất
a_tot = max(diem, key=diem.get)

# 8. Huấn luyện mô hình cuối cùng
m = mang((100, 100), a_tot).fit(X_tr, y_tr)

# 9. Đánh giá trên tập kiểm tra
print()
print("alpha tot nhat =", a_tot)

print(
    "Do chinh xac tap kiem tra:",
    f"{accuracy_score(y_te, m.predict(X_te)):.4f}"
)


Ba mang             Tap hoc    Tap kiem tra
(2,), alpha 1e-4    0.8444    0.8533
(100, 100), 1e-5    1.0000    0.7733
(100, 100), 1.0     0.8844    0.9067

Alpha kiem dinh cheo
1e-05    0.7822
0.001    0.7822


KeyboardInterrupt: 

In [66]:
"""Bước 4: mạng nơ-ron khi quên chuẩn hoá."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_mo_rong.csv")   # file nằm ở thư mục gốc của Colab
print(df.head())                            # xem tên cột; xóa dòng này khi đã ổn
print()

X = df[["gio_on", "diem_giua_ky", "chi_tieu"]]
y = df["qua_mon"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

tho = MLPClassifier(hidden_layer_sizes=(20,), max_iter=3000,
                    random_state=0).fit(X_tr, y_tr)
chuan = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(20,), max_iter=3000,
                  random_state=0),
).fit(X_tr, y_tr)

print(f"Doan bua ca lop qua : {y_te.mean():.4f}")
print(f"Khong chuan hoa     : "
      f"{accuracy_score(y_te, tho.predict(X_te)):.4f}")
print(f"Co chuan hoa        : "
      f"{accuracy_score(y_te, chuan.predict(X_te)):.4f}")
print()

# Mạng không chuẩn hoá đoán ra sao
du = tho.predict(X_te)
print("Khong chuan hoa, so ban doan qua:", du.sum(), "tren", len(du))

   gio_on  diem_giua_ky  chi_tieu  qua_mon
0     5.5           3.0      5050        0
1    19.0           6.0      7700        1
2    14.0           7.7      2850        0
3    11.0           3.1      7700        0
4    10.5           5.8      3850        1

Doan bua ca lop qua : 0.6000
Khong chuan hoa     : 0.4000
Co chuan hoa        : 0.9000

Khong chuan hoa, so ban doan qua: 0 tren 30


In [67]:
"""Bước 5: nhận dạng chữ số viết tay bằng MLP."""

import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

mang = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000,
                  random_state=0),
).fit(X_tr, y_tr)
mlp = mang[-1]

print("Kich thuoc cac ma tran trong so:",
      [w.shape for w in mlp.coefs_])
so_ts = sum(w.size for w in mlp.coefs_)
so_ts += sum(b.size for b in mlp.intercepts_)
print("Tong so tham so:", so_ts)
print("So vong hoc:", mlp.n_iter_)
mm = mlp.loss_curve_
print(f"Mat mat vong 1: {mm[0]:.4f}, vong 10: {mm[9]:.4f},"
      f" vong cuoi: {mm[-1]:.4f}")
print()

du = mang.predict(X_te)
print(f"Accuracy tren tap kiem tra: {accuracy_score(y_te, du):.4f}")
sai = np.flatnonzero(du != y_te)
print("So anh doan sai:", len(sai), "tren", len(y_te))
for i in sai:
    print(f"  anh {i:3d}: that la {y_te[i]}, doan la {du[i]}")
print()

# Dừng sớm: giữ riêng 10% tập học để theo dõi
dung_som = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000,
                  early_stopping=True, random_state=0),
).fit(X_tr, y_tr)
print("Dung som sau", dung_som[-1].n_iter_, "vong, accuracy",
      f"{accuracy_score(y_te, dung_som.predict(X_te)):.4f}")
print()

# So với các mô hình đã học, cùng tập học và tập kiểm tra
khac = {
    "Logistic co chuan hoa": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000)),
    "KNN, k = 5": KNeighborsClassifier(),
    "SVM kernel RBF": SVC(),
}
for ten, mh in khac.items():
    acc = accuracy_score(y_te, mh.fit(X_tr, y_tr).predict(X_te))
    print(f"{ten:22s} {acc:.4f}")

Kich thuoc cac ma tran trong so: [(64, 100), (100, 10)]
Tong so tham so: 7510
So vong hoc: 144
Mat mat vong 1: 2.5206, vong 10: 0.4448, vong cuoi: 0.0052

Accuracy tren tap kiem tra: 0.9844
So anh doan sai: 7 tren 450
  anh   6: that la 9, doan la 3
  anh  61: that la 5, doan la 7
  anh  77: that la 2, doan la 1
  anh  85: that la 9, doan la 7
  anh 276: that la 8, doan la 2
  anh 428: that la 7, doan la 9
  anh 443: that la 4, doan la 7

Dung som sau 44 vong, accuracy 0.9578

Logistic co chuan hoa  0.9711
KNN, k = 5             0.9867
SVM kernel RBF         0.9911


In [70]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_moons
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [71]:
df = pd.read_csv("xor.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

mlp = MLPClassifier(hidden_layer_sizes=(8,), solver="lbfgs",
                    max_iter=5000, random_state=0).fit(X_tr, y_tr)

def relu(z):
    return np.maximum(0, z)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

W1, W2 = mlp.coefs_
b1, b2 = mlp.intercepts_

X5 = X_te.iloc[:5].to_numpy(dtype=float)
H = relu(X5 @ W1 + b1)
tay = sigmoid(H @ W2 + b2).ravel()
sk = mlp.predict_proba(X_te.iloc[:5])[:, 1]

print("Tu tinh      :", np.round(tay, 10))
print("predict_proba:", np.round(sk, 10))
print("Khop nhau    :", np.allclose(tay, sk))

Tu tinh      : [0.01775032 0.99946016 0.79969969 0.13650667 0.95776552]
predict_proba: [0.01775032 0.99946016 0.79969969 0.13650667 0.95776552]
Khop nhau    : True


In [72]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
h1 = np.maximum(0, X[:, 0] - X[:, 1])
h2 = np.maximum(0, X[:, 1] - X[:, 0])
y = h1 + h2
print("x1 x2 h1 h2 y")
for xi, a, b, c in zip(X, h1, h2, y):
    print(f"{xi[0]:.0f}  {xi[1]:.0f}  {a:.0f}  {b:.0f}  {c:.0f}")
print("Dung XOR:", np.array_equal(y, np.array([0, 1, 1, 0])))

x1 x2 h1 h2 y
0  0  0  0  0
0  1  0  1  1
1  0  1  0  1
1  1  0  0  0
Dung XOR: True


In [73]:
df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("activation | tap hoc | tap kiem tra")
for act in ["logistic", "tanh", "relu"]:
    m = make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(100, 100), alpha=1.0,
                      activation=act, solver="lbfgs",
                      max_iter=20000, random_state=0),
    ).fit(X_tr, y_tr)
    print(f"{act:10s} | {accuracy_score(y_tr, m.predict(X_tr)):.4f}  | "
          f"{accuracy_score(y_te, m.predict(X_te)):.4f}")

activation | tap hoc | tap kiem tra
logistic   | 0.8400  | 0.8533
tanh       | 0.8711  | 0.9067
relu       | 0.8844  | 0.9067


In [76]:
X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

plt.figure(figsize=(8, 5))
print("lr_init | n_iter | loss cuoi | accuracy kiem tra")
for lr in [0.0001, 0.001, 0.01, 0.1]:
    m = make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(100,), learning_rate_init=lr,
                      max_iter=1000, random_state=0),
    ).fit(X_tr, y_tr)
    mlp = m[-1]
    acc = accuracy_score(y_te, m.predict(X_te))
    print(f"{lr:<7} | {mlp.n_iter_:6d} | {mlp.loss_curve_[-1]:.4f}    | {acc:.4f}")
    plt.plot(mlp.loss_curve_, label=f"lr = {lr}")

plt.xlabel("Vong hoc")
plt.ylabel("Mat mat")
plt.yscale("log")
plt.legend()
plt.title("Toc do hoc")
plt.savefig("toc_do_hoc.png", dpi=150, bbox_inches="tight")
plt.show()

lr_init | n_iter | loss cuoi | accuracy kiem tra
0.0001  |    582 | 0.0189    | 0.9889
0.001   |    144 | 0.0052    | 0.9844
0.01    |     38 | 0.0011    | 0.9778
0.1     |     30 | 0.1230    | 0.9667


In [74]:
mo_hinh = make_pipeline(
    StandardScaler(),
    MLPClassifier(max_iter=1000, random_state=0),
)
luoi = {
    "mlpclassifier__hidden_layer_sizes": [(10,), (50,), (100,), (50, 50)],
    "mlpclassifier__alpha": [0.0001, 0.01, 1],
}
gs = GridSearchCV(mo_hinh, luoi, cv=5, n_jobs=-1).fit(X_tr, y_tr)

print("Cap tot nhat:", gs.best_params_)
print(f"Kiem dinh cheo tot nhat: {gs.best_score_:.4f}")
print(f"Accuracy kiem tra (MLP): {accuracy_score(y_te, gs.predict(X_te)):.4f}")

svm = SVC().fit(X_tr, y_tr)
print(f"Accuracy kiem tra (SVM): {accuracy_score(y_te, svm.predict(X_te)):.4f}")

Cap tot nhat: {'mlpclassifier__alpha': 0.0001, 'mlpclassifier__hidden_layer_sizes': (50, 50)}
Kiem dinh cheo tot nhat: 0.8578
Accuracy kiem tra (MLP): 0.8800
Accuracy kiem tra (SVM): 0.9200


In [75]:
df = pd.read_csv("gia_nha.csv")
print(df.head())

COT_GIA = df.columns[-1]       # cot gia nha: sua neu khong phai cot cuoi
COT_DIEN_TICH = df.columns[0]  # cot dien tich (m2): sua cho dung ten cot

X = df.drop(columns=[COT_GIA])
y = df[COT_GIA]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17
)

tuyen_tinh = LinearRegression().fit(X_tr, y_tr)
mang = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(20,), max_iter=5000, random_state=0),
).fit(X_tr, y_tr)

print(f"MAE tuyen tinh: {mean_absolute_error(y_te, tuyen_tinh.predict(X_te)):.4f}")
print(f"MAE no-ron    : {mean_absolute_error(y_te, mang.predict(X_te)):.4f}")

# Can 250 m2: cac cot khac lay gia tri trung binh
dong = X.mean().to_frame().T
dong[COT_DIEN_TICH] = 250
print("Dien tich hoc trong khoang:", X[COT_DIEN_TICH].min(), "den", X[COT_DIEN_TICH].max())
print(f"Du doan 250 m2 - tuyen tinh: {tuyen_tinh.predict(dong)[0]:.4f}")
print(f"Du doan 250 m2 - no-ron    : {mang.predict(dong)[0]:.4f}")

   dien_tich  so_phong  tuoi_nha   gia
0       35.5         1        18  3.00
1       37.6         2         7  3.87
2       49.4         2        20  4.25
3       49.0         2        16  4.60
4       49.9         1        12  3.99
MAE tuyen tinh: 0.2185
MAE no-ron    : 0.4242
Dien tich hoc trong khoang: 35.5 den 117.5
Du doan 250 m2 - tuyen tinh: 18.6188
Du doan 250 m2 - no-ron    : 21.1072
